In [ ]:
import json
from pymongo import MongoClient
from datetime import datetime

In [6]:
MONGO_URI = "mongodb://localhost:27017"
DB_NAME = "weather_project"

client = MongoClient(MONGO_URI)
db = client[DB_NAME]

def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def migrate_station(file_path, collection_name):
    data = load_json(file_path)
    collection = db[collection_name]

    collection.drop()   # Reset collection

    # si liste -> insert_many
    if isinstance(data, list) and len(data) > 0:
        collection.insert_many(data)
        print(f" import OK ({len(data)} documents) -> {collection_name}")

    # si objet -> insert_one
    elif isinstance(data, dict):
        collection.insert_one(data)
        print(f" import OK (1 document) -> {collection_name}")

    else:
        raise ValueError(f" Format JSON non supporté dans {file_path}")


def quality_control(collection_name):
    col = db[collection_name]

    total = col.count_documents({})
    missing_location = col.count_documents({
        "$or": [
            {"location.latitude": None},
            {"location.longitude": None}
        ]
    })
    missing_name = col.count_documents({"name": None})

    print(f"\n--- QUALITÉ {collection_name} ---")
    print("Documents :", total)

    if total == 0:
        print("⚠ collection vide")
        return
    
    print("Taux erreurs coordonnées :", round((missing_location / total)*100,2), "%")
    print("Taux erreurs nom :", round((missing_name / total)*100,2), "%")


if __name__ == "__main__":

    print("===== MIGRATION LANCÉE =====")

    migrate_station("C:/Users/cheik/projet8/output/la_madeleine_mongo.json", "la_madeleine")
    migrate_station("C:/Users/cheik/projet8/output/ichtegem_mongo.json", "ichtegem")
    migrate_station("C:/Users/cheik/projet8/output/infoclimat_mongo.json", "infoclimat")

    quality_control("la_madeleine")
    quality_control("ichtegem")
    quality_control("infoclimat")

    print("\n FINI ")


===== MIGRATION LANCÉE =====
 import OK (1 document) -> la_madeleine
 import OK (1 document) -> ichtegem
 import OK (4 documents) -> infoclimat

--- QUALITÉ la_madeleine ---
Documents : 1
Taux erreurs coordonnées : 0.0 %
Taux erreurs nom : 0.0 %

--- QUALITÉ ichtegem ---
Documents : 1
Taux erreurs coordonnées : 0.0 %
Taux erreurs nom : 0.0 %

--- QUALITÉ infoclimat ---
Documents : 4
Taux erreurs coordonnées : 0.0 %
Taux erreurs nom : 0.0 %

 FINI 
